#Dúvidas para confirmar sobre esse algoritmo

Aqui eu estou recebendo SMILES e calculando os descritores no notebook, ou a entrada deve ser os descritores prontos? Escolhi seis inicialmente, tem algum conjunto específico que devo usar?

Tem alguma base indicada com moléculas rotuladas como ativas/inativas? Seria bom deixar uma base e um exemplo aqui ou so codigo generico msm?

Usei MinMax para normalizar. Posso manter ou seria melhor z-score?

Deixei uma camada oculta com quatro neurônios, divisão 80/20 e métricas de avaliação. posso deixar assim msm? 
Outra coisa, o artigo prevê energia de ligação, mas o repository pede classificação de atividade. Segui o que esta escrito no repository

# ANN Activity Classifier

This notebook trains an artificial neural network to classify molecular
activity using scikit-learn's MLPClassifier and molecular descriptors
calculated with RDKit.

The workflow includes descriptor calculation, training and test data
separation, feature scaling, model training, evaluation, and prediction
for new molecules.

In [3]:
import sys

import numpy as np
import pandas as pd
import rdkit
import sklearn

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from rdkit.Chem import Descriptors, Lipinski


## Input Data

The training dataset must be a CSV file containing:

- `smiles`: the molecular structure as a SMILES string.
- `activity`: the known activity class for that molecule.

Activity labels must correspond to the target and endpoint being studied.
Continuous measurements must be converted into classes using a
researcher-defined, scientifically justified criterion before training.

Each molecular structure must appear only once in the training dataset.
Molecules must follow consistent preparation conventions.

File paths and column names are defined in the configuration cell.

## Configuration

The configuration cell defines file paths, column names, molecular
descriptors, and model settings.

The default configuration uses six descriptors, one hidden layer with
four neurons, and a stratified 80/20 train-test split.

These settings can be adjusted for the dataset and research objective.
A fixed random seed controls the random split and model initialization.

In [4]:
TRAINING_FILE = "training.csv"
CANDIDATES_FILE = "candidates.csv"

SMILES_COLUMN = "smiles"
TARGET_COLUMN = "activity"

TEST_SIZE = 0.20
RANDOM_STATE = 42

HIDDEN_LAYER_SIZES = (4,)
ACTIVATION = "logistic"
SOLVER = "adam"
MAX_ITER = 1000

DESCRIPTOR_FUNCTIONS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "NumHDonors": Lipinski.NumHDonors,
    "NumHAcceptors": Lipinski.NumHAcceptors,
    "NumRotatableBonds": Lipinski.NumRotatableBonds,
}

## Molecular Descriptors

RDKit converts each molecule into a numerical descriptor vector.

| Descriptor | Description |
|---|---|
| MolWt | Molecular weight |
| MolLogP | Calculated octanol/water partition coefficient |
| TPSA | Topological polar surface area |
| NumHDonors | Number of hydrogen-bond donors |
| NumHAcceptors | Number of hydrogen-bond acceptors |
| NumRotatableBonds | Number of rotatable bonds |

The descriptor table is the input to the neural network.
Activity labels are the training targets.

Training and prediction must use the same descriptor definitions
and column order. Changing the descriptors requires retraining.

In [5]:
def calculate_descriptors(smiles_series):
    """Calculate the configured RDKit descriptors for each SMILES."""
    records = []

    for row_index, smiles in smiles_series.items():
        if not isinstance(smiles, str) or not smiles.strip():
            raise ValueError(f"Missing or empty SMILES at row {row_index}.")

        molecule = Chem.MolFromSmiles(smiles)

        if molecule is None or molecule.GetNumAtoms() == 0:
            raise ValueError(f"Invalid SMILES at row {row_index}: {smiles}")

        records.append({
            name: function(molecule)
            for name, function in DESCRIPTOR_FUNCTIONS.items()
        })

    descriptors = pd.DataFrame(
        records,
        index=smiles_series.index,
        columns=list(DESCRIPTOR_FUNCTIONS),
        dtype=float,
    )

    if not np.isfinite(descriptors.to_numpy()).all():
        raise ValueError("Descriptors contain missing or infinite values.")

    return descriptors

## Loading the Training Dataset

The following cell loads the training CSV and checks the required
columns, activity labels, and molecular structures.

Repeated structures are detected using canonical SMILES. Repeated
records and conflicting labels must be resolved before splitting
the dataset. This check does not replace molecular standardization.

`X` contains the molecular descriptors.
`y` contains the known activity classes.

A labeled training file is required to execute this and subsequent
data-dependent steps.

In [ ]:
training_data = pd.read_csv(
    TRAINING_FILE,
    dtype={
        SMILES_COLUMN: "string",
        TARGET_COLUMN: "string",
    },
)

required_columns = {SMILES_COLUMN, TARGET_COLUMN}
missing_columns = required_columns - set(training_data.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns: {sorted(missing_columns)}"
    )

if training_data.empty:
    raise ValueError("The training dataset is empty.")

labels = training_data[TARGET_COLUMN].str.strip()

if labels.isna().any() or labels.eq("").any():
    raise ValueError(
        "Activity labels must not be missing or empty."
    )

if labels.nunique() < 2:
    raise ValueError(
        "Classification requires at least two activity classes."
    )

if labels.value_counts().min() < 2:
    raise ValueError(
        "Each class needs at least two records for this split."
    )

X = calculate_descriptors(training_data[SMILES_COLUMN])
y = labels

canonical_smiles = training_data[SMILES_COLUMN].map(
    lambda smiles: Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
)

if canonical_smiles.duplicated().any():
    raise ValueError(
        "Duplicate molecular structures found. "
        "Resolve repeated records and conflicting labels "
        "before splitting."
    )

print("Number of molecules:", len(training_data))
print("\nClass counts:")
print(y.value_counts())

X.head()

## Model Training

The dataset is divided into training and test sets using a stratified
random split, approximately preserving the proportion of each class.
Both subsets must have enough samples to contain all classes.

A pipeline combines MinMaxScaler with MLPClassifier.
The scaler is fitted only on the training data and reused to transform
test data and new candidates.

Training descriptor values are scaled to the [0, 1] range.
Values from other datasets may fall outside that range.

The classifier learns the relationship between molecular descriptors
and activity labels from the training set.

If a convergence warning occurs, review training convergence before
interpreting the results. Model settings should not be tuned through
repeated inspection of test performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = Pipeline([
    ("scaler", MinMaxScaler()),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYER_SIZES,
        activation=ACTIVATION,
        solver=SOLVER,
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE,
    )),
])

model.fit(X_train, y_train)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print(
    "Training iterations:",
    model.named_steps["classifier"].n_iter_,
)

## Model Evaluation

The trained model predicts activity classes for the held-out test set.

The classification report presents precision, recall, F1-score,
and the number of test samples for each class.

The confusion matrix compares known and predicted classes:
rows represent known classes, and columns represent predictions.

This evaluation measures performance on the selected test set.
Closely related molecules may occur in both training and test sets.
Results do not establish generalization to new chemical scaffolds,
other targets, or other experimental settings.

In [ ]:
test_predictions = model.predict(X_test)
class_labels = model.classes_

print(classification_report(
    y_test,
    test_predictions,
    labels=class_labels,
    zero_division=0,
))

confusion = pd.DataFrame(
    confusion_matrix(
        y_test,
        test_predictions,
        labels=class_labels,
    ),
    index=pd.Index(class_labels, name="Actual"),
    columns=pd.Index(class_labels, name="Predicted"),
)

confusion

## Prediction for New Molecules

The candidate CSV must contain the configured SMILES column.
Activity labels are not required.

The notebook calculates the same descriptors used during training
and applies the fitted pipeline to predict activity classes.

Predictions are saved to `ann_predictions.csv`, preserving the input
columns and adding a `predicted_activity` column.

Predicted classes follow the activity definitions of the training
dataset and require experimental confirmation.

In [ ]:
def predict_activity(candidate_data, fitted_model):
    """Predict activity classes using a fitted descriptor-based pipeline."""
    if SMILES_COLUMN not in candidate_data.columns:
        raise ValueError(
            f"Missing column: {SMILES_COLUMN}"
        )

    if candidate_data.empty:
        raise ValueError("The candidate dataset is empty.")

    candidate_descriptors = calculate_descriptors(
        candidate_data[SMILES_COLUMN]
    )

    expected_columns = list(fitted_model.feature_names_in_)

    if list(candidate_descriptors.columns) != expected_columns:
        raise ValueError(
            "Descriptor columns differ from those used during training."
        )

    results = candidate_data.copy()

    results["predicted_activity"] = fitted_model.predict(
        candidate_descriptors
    )

    return results

In [ ]:
candidate_data = pd.read_csv(
    CANDIDATES_FILE,
    dtype={SMILES_COLUMN: "string"},
)

predictions = predict_activity(candidate_data, model)

predictions.to_csv(
    "ann_predictions.csv",
    index=False,
)

predictions.head()